In [1]:

import pandas as pd
import geopandas as gpd



dzielnice_gdf = gpd.read_file("dane/dzielnice_lublin_epsg2180.gml", encoding="UTF-8")
lokale_gdf = gpd.read_file("dane/lokale_lublin_epsg2180.gml", encoding="UTF-8")


In [2]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 2000)
pd.set_option('display.max_rows', 100)

In [3]:
lokale_gdf.head()

,gml_id,serwis_rcn,teryt,tran_przestrzen_nazw,tran_lokalny_id_iip,tran_wersja_id,tran_rodzaj_trans,tran_rodzaj_rynku,tran_sprzedajacy,tran_kupujacy,tran_cena_brutto,tran_vat,dok_data,nier_rodzaj,nier_prawo,nier_udzial,nier_pow_gruntu,nier_cena_brutto,nier_vat,lok_id_lokalu,lok_nr_lokalu,lok_funkcja,lok_liczba_izb,lok_nr_kond,lok_pow_uzyt,lok_pow_przyn,lok_cena_brutto,lok_vat,lok_adres,geometry
0,lokale.7652240,None,0609,PL.PZGiK.9349.RCN,5A4905BC-D6A2-40D6-8353-6E94AF99B6AD,NaN,wolnyRynek,NaN,osobaPrawna,osobaFizyczna,370000,NaN,2026-01-26 01:00:00+01,nieruchomoscLokalowa,wlasnoscLokaluWrazZPrawemZwiazanym,1/1,0.14,370000,NaN,060911_2.0007.494_BUD.1_LOK,1_LOK,mieszkalna,NaN,NaN,35.8,NaN,NaN,NaN,NaN,POINT (749233.476 386125.694)
1,lokale.7702785,None,0663,PL.PZGiK.4884.RCN,98781cfc-2bb7-4d85-9664-c969a4b2692f,2013-02-21T14:26:22,sprzedazBezprzetargowa,pierwotny,osobaPrawna,osobaFizyczna,20000,NaN,2012-09-27 02:00:00+02,nieruchomoscLokalowa,wlasnoscLokaluWrazZPrawemZwiazanym,1/1,NaN,20000,NaN,066301_1.0027.AR_12.34/1.1_BUD.33_LOK,33_LOK,garaz,1,0,396,NaN,20000,NaN,MSC:Lublin;UL:Kryształowa;NR_PORZ:30,POINT (744316.367 378735.781)
2,lokale.7691559,None,0663,PL.PZGiK.4884.RCN,dd4ba0a2-1bf5-4246-913b-643e6abbecad,2013-01-11T11:11:48,wolnyRynek,wtorny,osobaFizyczna,osobaFizyczna,160000,NaN,2012-09-24 02:00:00+02,nieruchomoscLokalowa,wlasnoscLokaluWrazZPrawemZwiazanym,1/1,NaN,160000,NaN,066301_1.0019.AR_9.9/8.1_BUD.1_LOK,1_LOK,mieszkalna,3,4,45.84,NaN,160000,NaN,MSC:Lublin;UL:Puchacza;NR_PORZ:8,POINT (750569.449 380204.84)
3,lokale.7682760,None,0663,PL.PZGiK.4884.RCN,33e4defe-db48-4760-8efd-ce127c4b8d75,2013-03-14T09:57:40,wolnyRynek,wtorny,osobaFizyczna,osobaFizyczna,257000,NaN,2012-12-07 01:00:00+01,nieruchomoscLokalowa,wlasnoscLokaluWrazZPrawemZwiazanym,1/1,NaN,257000,NaN,066301_1.0006.AR_5.1/4.13_BUD.1_LOK,1_LOK,mieszkalna,4,1,69.6,4.4,257000,NaN,MSC:Lublin;UL:Leszetyckiego;NR_PORZ:6,POINT (747737.827 384676.722)
4,lokale.7693503,None,0663,PL.PZGiK.4884.RCN,d2afa76b-927a-4325-9f0f-74c3c9fbc15a,2013-07-05T14:01:57,sprzedazBezprzetargowa,pierwotny,osobaPrawna,osobaFizyczna,225000,16666.67,2013-04-22 02:00:00+02,nieruchomoscLokalowa,wlasnoscLokaluWrazZPrawemZwiazanym,1/1,NaN,225000,16666.67,066301_1.0024.AR_7.12/2.1_BUD.86_LOK,86_LOK,mieszkalna,3,4,49.62,10.79,225000,16666.67,MSC:Lublin;UL:Strzeszewskiego;NR_PORZ:17,POINT (749805.079 384621.036)


In [4]:
dzielnice_gdf.head()

,gml_id,lowerCorner,upperCorner,nazwa,info,geometry
0,dzielnice_granice.1164,380959.864812533 745566.693777366,382835.984607483 747793.410854866,Wieniawa,None,"POLYGON ((747790.92 381981.663, 747785.876 382..."
1,dzielnice_granice.1168,381641.690850771 744139.041984043,382920.614087444 746343.181421421,Sławinek,None,"POLYGON ((746092.318 382325.297, 746149.849 38..."
2,dzielnice_granice.1174,381420.353809034 749298.987409915,383660.86866606 751731.326097459,Kalinowszczyzna,None,"POLYGON ((750171.505 381711.047, 750172.253 38..."
3,dzielnice_granice.1177,377485.315997601 741124.008056947,380376.543745426 744815.480524204,Węglin Pd.,None,"POLYGON ((744352.744 379570.608, 744315.008 37..."
4,dzielnice_granice.1186,379533.065232011 749325.021498149,381421.792408666 752180.424924755,Bronowice,None,"POLYGON ((752041.641 380648.706, 752180.425 38..."


In [5]:
# Funkcja do dodania kolumny dzielnica
def dodaj_dzielnice_do_lokali(gdf_lokale, gdf_dzielnice, nazwa_kolumny_dzielnica='dzielnica', nazwa_kolumny_nazwa='name'):
    """
    Dodaje kolumnę z nazwą dzielnicy do GeoDataFrame lokali na podstawie spatial join.
    
    Parametry:
    -----------
    gdf_lokale : GeoDataFrame
        GeoDataFrame z lokaliami, musi mieć kolumnę geometry (punkty)
    gdf_dzielnice : GeoDataFrame
        GeoDataFrame z dzielnicami, musi mieć kolumnę geometry (wielokąty)
    nazwa_kolumny_dzielnica : str
        Nazwa nowej kolumny z dzielnicą
    nazwa_kolumny_nazwa : str
        Nazwa kolumny w gdf_dzielnice zawierającej nazwy dzielnic
    
    Zwraca:
    --------
    GeoDataFrame
        GeoDataFrame z dodaną kolumną dzielnica
    """
    print(f"\nDodawanie kolumny '{nazwa_kolumny_dzielnica}' do lokali...")
    
    # Spatial join - dla każdego lokalu znajdź dzielnicę
    gdf_result = gpd.sjoin(gdf_lokale, gdf_dzielnice[['geometry', nazwa_kolumny_nazwa]], 
                            how='left', predicate='within')
    
    # Zmień nazwę kolumny
    if nazwa_kolumny_nazwa in gdf_result.columns:
        gdf_result[nazwa_kolumny_dzielnica] = gdf_result[nazwa_kolumny_nazwa]
        gdf_result = gdf_result.drop(columns=[nazwa_kolumny_nazwa])
    
    # Usuń index z spatial join
    if 'index_right' in gdf_result.columns:
        gdf_result = gdf_result.drop(columns=['index_right'])
    
    print(f"✓ Dodano kolumnę '{nazwa_kolumny_dzielnica}'")
    
    # Statystyki
    lokale_z_dzielnica = gdf_result[gdf_result[nazwa_kolumny_dzielnica].notna()].shape[0]
    lokale_bez_dzielnica = gdf_result[gdf_result[nazwa_kolumny_dzielnica].isna()].shape[0]
    
    print(f"  Lokale z przypisaną dzielnicą: {lokale_z_dzielnica}")
    print(f"  Lokale bez przypisanej dzielnicy: {lokale_bez_dzielnica}")
    
    if lokale_z_dzielnica > 0:
        print(f"\n  Rozkład lokali po dzielnicach:")
        dzielnice_count = gdf_result[nazwa_kolumny_dzielnica].value_counts()
        for dzielnica, count in dzielnice_count.items():
            print(f"    {dzielnica}: {count}")
    
    return gdf_result

combined_gdf = dodaj_dzielnice_do_lokali(lokale_gdf, dzielnice_gdf, "dzielnica", "nazwa")


Dodawanie kolumny 'dzielnica' do lokali...
✓ Dodano kolumnę 'dzielnica'
  Lokale z przypisaną dzielnicą: 96646
  Lokale bez przypisanej dzielnicy: 20

  Rozkład lokali po dzielnicach:
    Węglin Pd.: 12535
    Sławin: 8815
    Wrotków: 8441
    Ponikwoda: 8014
    Śródmieście: 7907
    Rury: 5326
    Dziesiąta: 5230
    Wieniawa: 5120
    Bronowice: 4834
    Czechów Pd.: 4632
    Kośminek: 4254
    Czuby Płn.: 3915
    Czuby Pd.: 3697
    Czechów Pł.: 3235
    Kalinowszczyzna: 2438
    Konstantynów: 2013
    Felin: 1935
    Tatary: 1918
    Szerokie: 787
    Sławinek: 764
    Za Cukrownią: 335
    Stare Miasto: 245
    Węglin Płn.: 83
    Abramowice: 75
    Hajdów - Zadębie: 42
    Zemborzyce: 31
    Głusk: 25


In [6]:
combined_gdf.head()

,gml_id,serwis_rcn,teryt,tran_przestrzen_nazw,tran_lokalny_id_iip,tran_wersja_id,tran_rodzaj_trans,tran_rodzaj_rynku,tran_sprzedajacy,tran_kupujacy,tran_cena_brutto,tran_vat,dok_data,nier_rodzaj,nier_prawo,nier_udzial,nier_pow_gruntu,nier_cena_brutto,nier_vat,lok_id_lokalu,lok_nr_lokalu,lok_funkcja,lok_liczba_izb,lok_nr_kond,lok_pow_uzyt,lok_pow_przyn,lok_cena_brutto,lok_vat,lok_adres,geometry,dzielnica
0,lokale.7652240,None,0609,PL.PZGiK.9349.RCN,5A4905BC-D6A2-40D6-8353-6E94AF99B6AD,NaN,wolnyRynek,NaN,osobaPrawna,osobaFizyczna,370000,NaN,2026-01-26 01:00:00+01,nieruchomoscLokalowa,wlasnoscLokaluWrazZPrawemZwiazanym,1/1,0.14,370000,NaN,060911_2.0007.494_BUD.1_LOK,1_LOK,mieszkalna,NaN,NaN,35.8,NaN,NaN,NaN,NaN,POINT (749233.476 386125.694),NaN
1,lokale.7702785,None,0663,PL.PZGiK.4884.RCN,98781cfc-2bb7-4d85-9664-c969a4b2692f,2013-02-21T14:26:22,sprzedazBezprzetargowa,pierwotny,osobaPrawna,osobaFizyczna,20000,NaN,2012-09-27 02:00:00+02,nieruchomoscLokalowa,wlasnoscLokaluWrazZPrawemZwiazanym,1/1,NaN,20000,NaN,066301_1.0027.AR_12.34/1.1_BUD.33_LOK,33_LOK,garaz,1,0,396,NaN,20000,NaN,MSC:Lublin;UL:Kryształowa;NR_PORZ:30,POINT (744316.367 378735.781),Węglin Pd.
2,lokale.7691559,None,0663,PL.PZGiK.4884.RCN,dd4ba0a2-1bf5-4246-913b-643e6abbecad,2013-01-11T11:11:48,wolnyRynek,wtorny,osobaFizyczna,osobaFizyczna,160000,NaN,2012-09-24 02:00:00+02,nieruchomoscLokalowa,wlasnoscLokaluWrazZPrawemZwiazanym,1/1,NaN,160000,NaN,066301_1.0019.AR_9.9/8.1_BUD.1_LOK,1_LOK,mieszkalna,3,4,45.84,NaN,160000,NaN,MSC:Lublin;UL:Puchacza;NR_PORZ:8,POINT (750569.449 380204.84),Bronowice
3,lokale.7682760,None,0663,PL.PZGiK.4884.RCN,33e4defe-db48-4760-8efd-ce127c4b8d75,2013-03-14T09:57:40,wolnyRynek,wtorny,osobaFizyczna,osobaFizyczna,257000,NaN,2012-12-07 01:00:00+01,nieruchomoscLokalowa,wlasnoscLokaluWrazZPrawemZwiazanym,1/1,NaN,257000,NaN,066301_1.0006.AR_5.1/4.13_BUD.1_LOK,1_LOK,mieszkalna,4,1,69.6,4.4,257000,NaN,MSC:Lublin;UL:Leszetyckiego;NR_PORZ:6,POINT (747737.827 384676.722),Czechów Pł.
4,lokale.7693503,None,0663,PL.PZGiK.4884.RCN,d2afa76b-927a-4325-9f0f-74c3c9fbc15a,2013-07-05T14:01:57,sprzedazBezprzetargowa,pierwotny,osobaPrawna,osobaFizyczna,225000,16666.67,2013-04-22 02:00:00+02,nieruchomoscLokalowa,wlasnoscLokaluWrazZPrawemZwiazanym,1/1,NaN,225000,16666.67,066301_1.0024.AR_7.12/2.1_BUD.86_LOK,86_LOK,mieszkalna,3,4,49.62,10.79,225000,16666.67,MSC:Lublin;UL:Strzeszewskiego;NR_PORZ:17,POINT (749805.079 384621.036),Ponikwoda


In [7]:
def rozdziel_adres(df, kolumna_zrodlowa='adres'):
    """
    Rozdziela kolumnę z adresem w formacie 'MSC:wartość;UL:wartość;NR_PORZ:wartość'
    na trzy kolumny: Miejscowość, Ulica, Numer
    
    Parametry:
    -----------
    df : DataFrame
        DataFrame do przetworzenia
    kolumna_zrodlowa : str
        Nazwa kolumny zawierającej adresy do rozdzielenia
    
    Zwraca:
    --------
    DataFrame
        DataFrame z trzema nowymi kolumnami: Miejscowość, Ulica, Numer
    """
    import pandas as pd
    
    df_copy = df.copy()
    
    # Inicjalizuj nowe kolumny z wartościami NaN
    df_copy['Miejscowość'] = None
    df_copy['Ulica'] = None
    df_copy['Numer'] = None
    
    # Iteruj po każdym rzędzie i rozdziel wartości
    for idx, row in df_copy.iterrows():
        if pd.notna(row[kolumna_zrodlowa]):
            adres_str = str(row[kolumna_zrodlowa])
            # Rozdziel po średniku
            pary = adres_str.split(';')
            
            for para in pary:
                if ':' in para:
                    klucz, wartość = para.split(':', 1)
                    klucz = klucz.strip()
                    wartość = wartość.strip()
                    
                    if klucz == 'MSC':
                        df_copy.at[idx, 'Miejscowość'] = wartość
                    elif klucz == 'UL':
                        df_copy.at[idx, 'Ulica'] = wartość
                    elif klucz == 'NR_PORZ':
                        df_copy.at[idx, 'Numer'] = wartość
    
    return df_copy

combined_gdf = rozdziel_adres(combined_gdf, "lok_adres")

In [8]:
combined_gdf.head()

,gml_id,serwis_rcn,teryt,tran_przestrzen_nazw,tran_lokalny_id_iip,tran_wersja_id,tran_rodzaj_trans,tran_rodzaj_rynku,tran_sprzedajacy,tran_kupujacy,tran_cena_brutto,tran_vat,dok_data,nier_rodzaj,nier_prawo,nier_udzial,nier_pow_gruntu,nier_cena_brutto,nier_vat,lok_id_lokalu,lok_nr_lokalu,lok_funkcja,lok_liczba_izb,lok_nr_kond,lok_pow_uzyt,lok_pow_przyn,lok_cena_brutto,lok_vat,lok_adres,geometry,dzielnica,Miejscowość,Ulica,Numer
0,lokale.7652240,None,0609,PL.PZGiK.9349.RCN,5A4905BC-D6A2-40D6-8353-6E94AF99B6AD,NaN,wolnyRynek,NaN,osobaPrawna,osobaFizyczna,370000,NaN,2026-01-26 01:00:00+01,nieruchomoscLokalowa,wlasnoscLokaluWrazZPrawemZwiazanym,1/1,0.14,370000,NaN,060911_2.0007.494_BUD.1_LOK,1_LOK,mieszkalna,NaN,NaN,35.8,NaN,NaN,NaN,NaN,POINT (749233.476 386125.694),NaN,None,None,None
1,lokale.7702785,None,0663,PL.PZGiK.4884.RCN,98781cfc-2bb7-4d85-9664-c969a4b2692f,2013-02-21T14:26:22,sprzedazBezprzetargowa,pierwotny,osobaPrawna,osobaFizyczna,20000,NaN,2012-09-27 02:00:00+02,nieruchomoscLokalowa,wlasnoscLokaluWrazZPrawemZwiazanym,1/1,NaN,20000,NaN,066301_1.0027.AR_12.34/1.1_BUD.33_LOK,33_LOK,garaz,1,0,396,NaN,20000,NaN,MSC:Lublin;UL:Kryształowa;NR_PORZ:30,POINT (744316.367 378735.781),Węglin Pd.,Lublin,Kryształowa,30
2,lokale.7691559,None,0663,PL.PZGiK.4884.RCN,dd4ba0a2-1bf5-4246-913b-643e6abbecad,2013-01-11T11:11:48,wolnyRynek,wtorny,osobaFizyczna,osobaFizyczna,160000,NaN,2012-09-24 02:00:00+02,nieruchomoscLokalowa,wlasnoscLokaluWrazZPrawemZwiazanym,1/1,NaN,160000,NaN,066301_1.0019.AR_9.9/8.1_BUD.1_LOK,1_LOK,mieszkalna,3,4,45.84,NaN,160000,NaN,MSC:Lublin;UL:Puchacza;NR_PORZ:8,POINT (750569.449 380204.84),Bronowice,Lublin,Puchacza,8
3,lokale.7682760,None,0663,PL.PZGiK.4884.RCN,33e4defe-db48-4760-8efd-ce127c4b8d75,2013-03-14T09:57:40,wolnyRynek,wtorny,osobaFizyczna,osobaFizyczna,257000,NaN,2012-12-07 01:00:00+01,nieruchomoscLokalowa,wlasnoscLokaluWrazZPrawemZwiazanym,1/1,NaN,257000,NaN,066301_1.0006.AR_5.1/4.13_BUD.1_LOK,1_LOK,mieszkalna,4,1,69.6,4.4,257000,NaN,MSC:Lublin;UL:Leszetyckiego;NR_PORZ:6,POINT (747737.827 384676.722),Czechów Pł.,Lublin,Leszetyckiego,6
4,lokale.7693503,None,0663,PL.PZGiK.4884.RCN,d2afa76b-927a-4325-9f0f-74c3c9fbc15a,2013-07-05T14:01:57,sprzedazBezprzetargowa,pierwotny,osobaPrawna,osobaFizyczna,225000,16666.67,2013-04-22 02:00:00+02,nieruchomoscLokalowa,wlasnoscLokaluWrazZPrawemZwiazanym,1/1,NaN,225000,16666.67,066301_1.0024.AR_7.12/2.1_BUD.86_LOK,86_LOK,mieszkalna,3,4,49.62,10.79,225000,16666.67,MSC:Lublin;UL:Strzeszewskiego;NR_PORZ:17,POINT (749805.079 384621.036),Ponikwoda,Lublin,Strzeszewskiego,17


In [9]:
combined_gdf.query("Ulica.str.contains('Balcera', na=False)")

,gml_id,serwis_rcn,teryt,tran_przestrzen_nazw,tran_lokalny_id_iip,tran_wersja_id,tran_rodzaj_trans,tran_rodzaj_rynku,tran_sprzedajacy,tran_kupujacy,tran_cena_brutto,tran_vat,dok_data,nier_rodzaj,nier_prawo,nier_udzial,nier_pow_gruntu,nier_cena_brutto,nier_vat,lok_id_lokalu,lok_nr_lokalu,lok_funkcja,lok_liczba_izb,lok_nr_kond,lok_pow_uzyt,lok_pow_przyn,lok_cena_brutto,lok_vat,lok_adres,geometry,dzielnica,Miejscowość,Ulica,Numer
151,lokale.7708656,None,0663,PL.PZGiK.4884.RCN,0eec0d6d-3cdb-4fa9-9231-bd08dab82747,2014-10-22T14:59:24,sprzedazBezprzetargowa,pierwotny,osobaPrawna,osobaFizyczna,252076,18672.3,2014-07-10 02:00:00+02,nieruchomoscLokalowa,wlasnoscLokaluWrazZPrawemZwiazanym,1/1,NaN,252076,18672.3,066301_1.0028.AR_2.34/219.1_BUD.24_LOK,24_LOK,mieszkalna,2,16,37.34,NaN,252076,18672.3,MSC:Lublin;UL:Pana Balcera;NR_PORZ:6,POINT (746113.241 380374.689),Rury,Lublin,Pana Balcera,6
253,lokale.7722703,None,0663,PL.PZGiK.4884.RCN,5718146a-2935-4849-8507-1b47e663cd71,2019-01-17T10:38:05,sprzedazBezprzetargowa,pierwotny,osobaPrawna,osobaFizyczna,221388,18170,2018-09-04 02:00:00+02,nieruchomoscLokalowa,wlasnoscLokaluWrazZPrawemZwiazanym,1/1,6147,205702,15237,066301_1.0028.AR_2.34/8.1_BUD.55_LOK,55_LOK,mieszkalna,2,3,31.34,NaN,NaN,NaN,MSC:Lublin;UL:Pana Balcera;NR_PORZ:6b,POINT (746197.79 380423.142),Rury,Lublin,Pana Balcera,6b
334,lokale.7709161,None,0663,PL.PZGiK.4884.RCN,b400a8ba-b151-4638-afac-296870d495fc,2015-02-05T11:19:52,sprzedazBezprzetargowa,pierwotny,osobaPrawna,osobaFizyczna,5000,934.96,2015-01-30 01:00:00+01,nieruchomoscLokalowa,wlasnoscLokaluWrazZPrawemZwiazanym,1/88,NaN,5000,934.96,066301_1.0028.AR_2.34/219.1_BUD.24_LOK,24_LOK,garaz,1,0,1100,NaN,5000,934.96,MSC:Lublin;UL:Pana Balcera;NR_PORZ:6,POINT (746113.241 380374.689),Rury,Lublin,Pana Balcera,6
369,lokale.7722544,None,0663,PL.PZGiK.4884.RCN,d548c8c6-d464-4a4e-9c3a-a9850cea7d77,2019-05-15T11:24:36,sprzedazBezprzetargowa,pierwotny,osobaPrawna,osobaFizyczna,517996,96861,2019-01-25 01:00:00+01,nieruchomoscLokalowa,wlasnoscLokaluWrazZPrawemZwiazanym,1/1,3655,517996,96861,066301_1.0028.AR_2.34/8.1_BUD.115_LOK,115_LOK,NaN,0,1,76.57,NaN,NaN,NaN,MSC:Lublin;UL:Pana Balcera;NR_PORZ:6b,POINT (746197.79 380423.142),Rury,Lublin,Pana Balcera,6b
600,lokale.7701900,None,0663,PL.PZGiK.4884.RCN,2e8a5deb-4585-4c41-983e-15502004ada5,2010-05-10T13:27:18,wolnyRynek,wtorny,osobaFizyczna,osobaFizyczna,165000,NaN,2010-03-15 01:00:00+01,nieruchomoscLokalowa,wlasnoscLokaluWrazZPrawemZwiazanym,1/1,NaN,165000,NaN,066301_1.0028.AR_2.21/2.1_BUD.3_LOK,3_LOK,mieszkalna,3,5,32,1.94,165000,NaN,MSC:Lublin;UL:Pana Balcera;NR_PORZ:12,POINT (746239.167 380070.713),Rury,Lublin,Pana Balcera,12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95470,lokale.7746335,None,0663,PL.PZGiK.4884.RCN,6dd6fba7-bddf-4842-a259-aef6d3c05b8a,2023-04-25T08:33:15,wolnyRynek,wtorny,osobaFizyczna,osobaFizyczna,260000,0,2023-03-03 01:00:00+01,nieruchomoscLokalowa,wlasnoscLokaluWrazZPrawemZwiazanym,1/1,21120,260000,0,066301_1.0028.AR_2.19/3.16_BUD.1_LOK,1_LOK,mieszkalna,3,1,32,1.85,NaN,NaN,MSC:Lublin;UL:Pana Balcera;NR_PORZ:1,POINT (745978.74 380389.254),Rury,Lublin,Pana Balcera,1
95531,lokale.7679746,None,0663,PL.PZGiK.4884.RCN,3fc80af4-0b1a-49bc-b557-07efda77e2d0,2014-07-02T11:32:13,sprzedazBezprzetargowa,pierwotny,osobaPrawna,osobaPrawna,749000,55481.48,2014-02-25 01:00:00+01,nieruchomoscLokalowa,wlasnoscLokaluWrazZPrawemZwiazanym,1/1,NaN,749000,55481.48,066301_1.0028.AR_2.34/219.1_BUD.24_LOK,24_LOK,mieszkalna,2,1,117.4,NaN,749000,55481.48,MSC:Lublin;UL:Pana Balcera;NR_PORZ:6,POINT (746113.241 380374.689),Rury,Lublin,Pana Balcera,6
95647,lokale.7672337,None,0663,PL.PZGiK.4884.RCN,6c607c10-f130-4075-bced-a1d4a9072138,2016-03-10T13:38:37,sprzedazBezprzetargowa,pierwotny,osobaPrawna,osobaFizyczna,335000,24814,2016-01-22 01:00:00+01,nieruchomoscLokalowa,wlasnoscLokaluWrazZPrawemZwiazanym,1/1,5035,335000,24814.81,066301_1.0028.AR_2.34/219.1

In [10]:
combined_gdf.to_file('lublin_data_epsg2180.gml', driver='GML')